In [1]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

df=pd.read_csv("data/titanic_clean.csv")

In [2]:
df["Sex_num"] = df["Sex"].map({"male": 0, "female": 1})
df = pd.concat([df, pd.get_dummies(df["Embarked"], prefix="Emb")], axis=1)
df["Title"] = df["Name"].str.extract(r",\s*([^.]+)\.")
rare = ["Dr","Rev","Major","Col","Mlle","the Countess","Capt",
        "Ms","Sir","Lady","Mme","Don","Jonkheer"]
df["Title"] = df["Title"].replace(rare, "Rare")
df = pd.concat([df, pd.get_dummies(df["Title"], prefix="Title")], axis=1)
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

cols = (["Pclass","Sex_num","Age","SibSp","Parch","Fare","HasCabin",
         "Emb_C","Emb_Q","Emb_S","FamilySize"]
        + [c for c in df.columns if c.startswith("Title_")])

X, y = df[cols], df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [3]:
##网络搜索

#定义“试那些参数”

param_grid={
    "max_depth":[3,4,5,7],
    "n_estimators":[100,300],
    "min_samples_leaf":[1,2],
}

#造一个GridSearchCV
grid=GridSearchCV(
    estimator=RandomForestClassifier(random_state=42,n_jobs=1),   #"你要调哪个模型"     一个还没学会怎么学的学生
    param_grid=param_grid,    #"我想试哪些参数组合"     给学生的几套学习方法
    cv=5,                #	"每套方法都用 5 折交叉验证评分"        每套方法都考 5 次试，取平均          
    scoring="accuracy",  #用准确率评分    用准确率当考试分数
    n_jobs=1,            #Windows上映1，防止多进程出错   "用几个 CPU 核同时跑"  请几个人同时批卷
    verbose=1,           #打印进度   批卷时报个进度，让你知道没卡死
)

#开始搜索
grid.fit(X_train,y_train)  #只能传训练集，不然传所有数据会导致数据泄露
#⭐ 为什么非要"先造、再 fit"？
##因为造对象的时候，你只是在"配置规则"；真正干活的是 .fit()。

#看结果
print("最佳参数：",grid.best_params_)   ##grid.best_params_	最佳参数组合	告诉你该用哪组参数
print("最佳cv分数:",round(grid.best_score_,4))  ##最佳 CV 分数	训练阶段的成绩（调参时用）  4是保留四位小数
print("测试集分数：",round(grid.score(X_test,y_test),4))  ##grid.score(X_test, y_test)	测试集分数	最终报告的成绩
print("最佳模型:", grid.best_estimator_)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


最佳参数： {'max_depth': 7, 'min_samples_leaf': 2, 'n_estimators': 300}
最佳cv分数: 0.8509
测试集分数： 0.7809
最佳模型: RandomForestClassifier(max_depth=7, min_samples_leaf=2, n_estimators=300,
                       n_jobs=1, random_state=42)


为什么cv分数通常比测试集分数高
测试集是"真考"，CV 是"模拟考"。
模拟考可以考很多次，真考只有一次。
模拟考的最高分，通常高于真考成绩。

In [4]:
results=pd.DataFrame(grid.cv_results_)
print(results[["param_max_depth","param_n_estimators","param_min_samples_leaf","mean_test_score","rank_test_score"]].sort_values("rank_test_score").head(10))

    param_max_depth  param_n_estimators  param_min_samples_leaf  \
15                7                 300                       2   
13                7                 300                       1   
14                7                 100                       2   
12                7                 100                       1   
8                 5                 100                       1   
4                 4                 100                       1   
5                 4                 300                       1   
6                 4                 100                       2   
7                 4                 300                       2   
9                 5                 300                       1   

    mean_test_score  rank_test_score  
15         0.850901                1  
13         0.849512                2  
14         0.848094                3  
12         0.843879                4  
8          0.839653                5  
4          0.838255         

In [5]:
from sklearn.model_selection import RandomizedSearchCV

random_search=RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42,n_jobs=1),
    param_distributions={
        "max_depth": [3, 4, 5, 7, 10, None],
        "n_estimators": [50, 100, 200, 300, 500],
        "min_samples_leaf": [1, 2, 3, 4],
    },
    n_iter=20,##随机测20组，在组合多的时候会比GridSearchCV快，但是结果找到最优只能找个大概，在参数多的时候用
    cv=5,
    random_state=42,
    n_jobs=1,
)
random_search.fit(X_train,y_train)

print()
print("随机搜索最佳参数:", random_search.best_params_)
print("随机搜索最佳 CV:", round(random_search.best_score_, 4))
print("随机搜索测试集:", round(random_search.score(X_test, y_test), 4))



随机搜索最佳参数: {'n_estimators': 300, 'min_samples_leaf': 2, 'max_depth': 10}
随机搜索最佳 CV: 0.8509
随机搜索测试集: 0.7809


In [6]:
import time

# ⚠️ 用小网格对比，避免跑太久把内核累死
# （上次用 120 组 × 5 折 = 600 次训练，跑了 348 秒后内核崩了）
small_grid = {
    "max_depth": [3, 5, 7],
    "n_estimators": [100, 300],
}

# GridSearchCV：3 × 2 = 6 组 × 5 折 = 30 次训练
t0 = time.time()
g = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=1),
    small_grid, cv=5, n_jobs=1
)
g.fit(X_train, y_train)
t_grid = time.time() - t0

# RandomizedSearchCV：随机抽 3 组 × 5 折 = 15 次训练
t0 = time.time()
r = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=1),
    small_grid, n_iter=3, cv=5, random_state=42, n_jobs=1
)
r.fit(X_train, y_train)
t_rand = time.time() - t0

print(f"GridSearchCV（6 组）耗时:        {t_grid:.1f} 秒")
print(f"RandomizedSearchCV（3 组）耗时:  {t_rand:.1f} 秒")
print(f"后者快了约 {t_grid / t_rand:.1f} 倍")


GridSearchCV（6 组）耗时:        5.4 秒
RandomizedSearchCV（3 组）耗时:  3.2 秒
后者快了约 1.7 倍


	    GridSearchCV	    RandomizedSearchCV
怎么试	所有组合都试	        随机抽 n_iter 组
组合多时	慢到爆炸	            快
结果	一定找到网格内最优	大概率接近最优
什么时候用	参数少（≤3 个）	参数多

## 网格搜索结论

### 1. best_score_ 和测试集分数的区别

best_score_ 是交叉验证的最佳分数（训练阶段），
score(X_test, y_test) 是测试集分数（最终成绩）。

前者通常更高，原因：
① 它是从多组参数里【挑出来的最大值】—— 取最大值天然偏高
② CV 分数本身有随机性 —— 挑最高分也就是挑了运气最好的
③ 测试集完全没参与调参 —— 它是"真考"，CV 是"模拟考"

这叫 Optimization Bias（优化偏差）。
差距 1~3% 正常，超过 5% 说明过拟合。

### 2. 为什么只能传训练集

GridSearchCV 内部已经做了交叉验证。
如果传全部数据，测试集就会参与"选参数"，
相当于提前看了考卷，分数会虚高。

这和"标准化时测试集只能用 transform 不能用 fit_transform"
是同一个道理：测试集必须保持干净，只能最后用一次。

### 3. 为什么参数多用 RandomizedSearchCV

GridSearchCV 试遍所有组合，参数多了会爆炸：
4 个参数（6×5×4×3）= 360 组 × 5 折 = 1800 次训练

RandomizedSearchCV 只随机抽 n_iter 组：
20 组 × 5 折 = 100 次训练，快 18 倍

随机抽也能找到好结果，因为"好区域"通常不小，
且大多数参数影响不大。代价是不保证绝对最优。

### 4. ⭐ 一个真实案例：CV 和测试集差了 7 个百分点

我的 GridSearchCV 选出：

| 项目 | 数值 |
| --- | --- |
| 最佳参数 | max_depth=7, n_estimators=300, min_samples_leaf=2 |
| 最佳 CV 分数 | 0.8509 |
| 测试集分数 | 0.7809 |
| **差距** | **7.0 个百分点** |

按判断标准（1~3% 正常，>5% 严重），这属于**严重过拟合**。

这说明三件事：

1. **Optimization Bias 真实存在** —— CV 分数是"16 组里挑出来的最大值"，天然偏高
2. **模型确实过拟合了** —— max_depth=7 在第七课就已被证明偏深（测试集只有 0.7865）
3. **数据量太小（889 行）** —— CV 分数波动大，很容易挑到"运气好"的组合

**更值得注意的一点**：
GridSearchCV 选出的"最佳参数"，测试集只有 0.7809，
反而**不如第七课手动调的 max_depth=5（0.8146）**。

→ **自动调参不是万能的。**
→ **在小数据集上，交叉验证选出来的"最优"，可能只是"最幸运"。**
→ **所以一定要看 best_score_ 和测试分数的差距。**
